[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_22_Cost_Engineering.ipynb)

# Lesson 22 — Cost Engineering for Production AI Agents

> **Phase 3 · Lesson 22 of 23.** Previous lesson: *Agent Frameworks (CrewAI / AutoGen / LangGraph)*. Next lesson: the **Phase 3 Capstone** — AutoResearcher v1.0 deployed, evaluated, secured, *and now cost-engineered*.

---

## Why cost engineering matters

You've shipped agents. They work. The bill arrives. 😬

Cost engineering is the discipline of making AI systems **financially sustainable** without degrading quality. It is the difference between a cool demo and a real product. Five practical levers move the needle in production:

| Lever | Typical savings | When it helps |
|---|---|---|
| **Model routing** (small ↔ large) | 5×–20× cheaper | Mixed-difficulty traffic |
| **Prompt caching** (`cache_control`) | up to **90%** off cached tokens | Long, stable system prompts / tools / docs |
| **Message Batches API** | **50%** off everything | Async / non-real-time work |
| **Token compression** | 30–70% input reduction | Long-running agents, big RAG contexts |
| **Per-tenant observability** | unlocks billing & limits | Multi-tenant SaaS |

By the end of this lesson you will:

1. Read an Anthropic invoice line-by-line and predict the next one.
2. Build a **router** that sends easy work to Haiku and hard work to Sonnet/Opus.
3. Use `cache_control` to cache a 10k-token system prompt and verify the discount.
4. Submit a **Message Batch** and pick up results asynchronously.
5. Compress long contexts losslessly-ish.
6. Wrap an agent in a **`CostMeter`** that attributes every dollar to a tenant.
7. Mini-capstone: a `CostAwareAutoResearcher` that combines all five levers.

> 💡 **Newbie framing:** Think of LLM cost the way you'd think about cloud cost. You don't pay for *features*, you pay for **tokens** — chunks of text the model reads (input) and writes (output). Different rooms (models) have different per-token rates. Caching is a fridge. Batching is overnight FedEx. Routing is "send the intern, not the partner." Everything else is bookkeeping.



## 0) Setup

Run the cell below once. It installs the Anthropic SDK + a tiny tokenizer + helpers, and loads your API key from **Colab Secrets** (the 🔑 icon in Colab's left sidebar — add a secret named `ANTHROPIC_API_KEY`).


In [ ]:
# 0) Install
!pip install -q anthropic tiktoken pandas matplotlib

# Load API key from Colab Secrets (preferred) or env var
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded from Colab Secrets ✓")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), (
        "Set ANTHROPIC_API_KEY in Colab Secrets OR export it in your shell."
    )
    print("API key loaded from env ✓")

from anthropic import Anthropic
client = Anthropic()
print("Anthropic client ready ✓")


---
## 1) The cost model — what you actually pay for

Anthropic charges per **million tokens (MTok)**, split into four buckets:

| Bucket | Meaning |
|---|---|
| **Input tokens** | Tokens the model *reads* (your system prompt + messages + tool definitions). |
| **Output tokens** | Tokens the model *writes*. Always more expensive than input. |
| **Cache write** | First-time write of a cached block. ~25% **more** than normal input. |
| **Cache read** | Re-reading a cached block. ~**10% of normal input** (= 90% off). |

The exact prices change; what matters is the **shape**:

- Output > Input (often 3–5×).
- Bigger model > smaller model (often 10–30×).
- Cache read ≪ Cache write < Input < Output.
- Batched everything = half price.

We'll encode the current published list prices below so you have a reusable cost calculator. Tweak the dict as Anthropic updates prices.


In [ ]:
# 1) Pricing table ($ per 1M tokens). Update as prices change.
# These are list prices for the public API at time of writing.
PRICES = {
    # model: {"input": $/MTok, "output": $/MTok, "cache_write": $/MTok, "cache_read": $/MTok}
    "claude-haiku-4-5-20251001":  {"input": 1.00,  "output": 5.00,  "cache_write": 1.25,  "cache_read": 0.10},
    "claude-sonnet-4-5":          {"input": 3.00,  "output": 15.00, "cache_write": 3.75,  "cache_read": 0.30},
    "claude-opus-4-5":            {"input": 15.00, "output": 75.00, "cache_write": 18.75, "cache_read": 1.50},
}

def cost_of(usage: dict, model: str, batched: bool = False) -> float:
    """Compute $ for a single call given an Anthropic-style usage dict.

    `usage` is what the SDK returns under `response.usage`. Keys we care about:
      input_tokens, output_tokens, cache_creation_input_tokens, cache_read_input_tokens.
    """
    p = PRICES[model]
    discount = 0.5 if batched else 1.0
    inp  = (usage.get("input_tokens", 0) or 0) * p["input"]
    out  = (usage.get("output_tokens", 0) or 0) * p["output"]
    cw   = (usage.get("cache_creation_input_tokens", 0) or 0) * p["cache_write"]
    cr   = (usage.get("cache_read_input_tokens", 0) or 0) * p["cache_read"]
    total_per_mtok = inp + out + cw + cr
    return discount * total_per_mtok / 1_000_000

# 💡 EXPERIMENT: Change `batched=True` and see the bill halve.
sample_usage = {
    "input_tokens": 12_000,
    "output_tokens": 800,
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
}
for m in PRICES:
    print(f"{m:>32s}  ${cost_of(sample_usage, m):.5f}")


**Reading this output:** the same 12k-in / 800-out workload costs ~30× more on Opus than on Haiku. That's the gap routing will close.

---
## 2) Measure before you optimize — a tiny cost tracker

Premature optimization is the root of all (cost) evil. Before doing *anything* clever, **instrument**. The wrapper below records cost, latency, model, and a free-form `tag` for every call. We'll reuse it everywhere downstream.


In [ ]:
# 2) Tracked client — wraps Anthropic with cost+latency logging
import time, json
from dataclasses import dataclass, field, asdict
from typing import Any

@dataclass
class CallRecord:
    model: str
    tag: str
    tenant: str
    latency_s: float
    cost_usd: float
    usage: dict
    batched: bool = False

class CostMeter:
    def __init__(self, client):
        self.client = client
        self.records: list[CallRecord] = []

    def messages_create(self, *, tag: str = "", tenant: str = "default",
                        batched: bool = False, **kwargs):
        """Drop-in for client.messages.create() that logs cost."""
        t0 = time.time()
        resp = self.client.messages.create(**kwargs)
        dt = time.time() - t0
        usage = resp.usage.model_dump() if hasattr(resp.usage, "model_dump") else dict(resp.usage)
        rec = CallRecord(
            model=kwargs["model"],
            tag=tag,
            tenant=tenant,
            latency_s=round(dt, 3),
            cost_usd=round(cost_of(usage, kwargs["model"], batched=batched), 6),
            usage=usage,
            batched=batched,
        )
        self.records.append(rec)
        return resp

    def total(self) -> float:
        return round(sum(r.cost_usd for r in self.records), 6)

    def by_tenant(self) -> dict[str, float]:
        out: dict[str, float] = {}
        for r in self.records:
            out[r.tenant] = round(out.get(r.tenant, 0.0) + r.cost_usd, 6)
        return out

    def by_tag(self) -> dict[str, float]:
        out: dict[str, float] = {}
        for r in self.records:
            out[r.tag] = round(out.get(r.tag, 0.0) + r.cost_usd, 6)
        return out

    def as_df(self):
        import pandas as pd
        return pd.DataFrame([asdict(r) for r in self.records])

meter = CostMeter(client)
print("CostMeter ready ✓")


> 💡 **Why this matters more than it looks:** the same wrapper will eventually power your billing pipeline, your per-tenant rate limits, your "did this PR make us more expensive?" CI check, and your finance team's Looker dashboard. The earlier you adopt it, the less retrofit pain later.

---
## 3) Model routing — small for easy, big for hard

**The single highest-leverage cost lever** in agent systems. Most production traffic is mixed: 80% of requests are easy (classification, extraction, short rewrites) and 20% need real reasoning. Sending everything to your biggest model is like flying first-class to the corner store.

The routing pattern in three steps:

1. **Classify** the incoming request as `easy` / `hard`.
2. **Dispatch** to a cheap or expensive model accordingly.
3. **Escalate** if the cheap model's answer fails a quality check.

The classifier is itself an LLM call — but a tiny one. Net savings still huge.


In [ ]:
# 3a) The router
ROUTING_MODELS = {
    "easy": "claude-haiku-4-5-20251001",
    "hard": "claude-sonnet-4-5",
}

CLASSIFIER_SYSTEM = """You are a request-difficulty classifier.
Reply with exactly one word: easy or hard.
- easy = factual lookup, short rewrite, classification, simple extraction, casual chat.
- hard = multi-step reasoning, code design, long analysis, ambiguous tradeoffs."""

def classify(task: str, tenant: str = "default") -> str:
    resp = meter.messages_create(
        tag="route_classify", tenant=tenant,
        model="claude-haiku-4-5-20251001",
        max_tokens=4,
        system=CLASSIFIER_SYSTEM,
        messages=[{"role": "user", "content": task}],
    )
    label = resp.content[0].text.strip().lower()
    return "hard" if "hard" in label else "easy"

def route_and_answer(task: str, tenant: str = "default") -> tuple[str, str]:
    difficulty = classify(task, tenant=tenant)
    model = ROUTING_MODELS[difficulty]
    resp = meter.messages_create(
        tag=f"answer_{difficulty}", tenant=tenant,
        model=model,
        max_tokens=400,
        messages=[{"role": "user", "content": task}],
    )
    return difficulty, resp.content[0].text.strip()


In [ ]:
# 3b) Try the router on a mixed batch
tasks = [
    "What's the capital of Bhutan?",
    "Translate 'good morning' to Italian.",
    "Design a sharding strategy for a write-heavy ledger across 3 regions, with failure-mode analysis.",
    "Summarize this sentence in 5 words: the quick brown fox jumps over the lazy dog.",
    "Refactor a 4-class inheritance hierarchy into composition; explain the tradeoffs.",
]

for t in tasks:
    diff, ans = route_and_answer(t)
    print(f"[{diff.upper():4s}] {t[:60]}{'…' if len(t)>60 else ''}")
    print(f"        → {ans[:100]}{'…' if len(ans)>100 else ''}\n")

print("Spend so far:", meter.by_tag())
print("Total:", f"${meter.total():.6f}")


> 💡 **EXPERIMENT:** Remove the classifier and send all 5 tasks to Sonnet. Compare totals. Then send them all to Haiku and compare quality on the *hard* ones. The router is the negotiated middle.

> ⚠️ **Pitfall:** classifier accuracy matters. If the classifier is wrong 20% of the time and routes hard work to Haiku, quality drops invisibly. Always pair a router with **evals** (Lesson 17) — measure quality per route, not just average cost.

> 🧠 **Escalation pattern:** for high-stakes routes you can let Haiku take a first pass and only escalate to Sonnet if a self-evaluation step returns `"low_confidence"`. Same cheap-first idea, smarter fallback.


---
## 4) Prompt caching with `cache_control` breakpoints

When you call Claude, anything you send (system prompt, tool defs, message history) is re-tokenized and re-charged **every call**. That's wasteful if those parts don't change between turns.

**Prompt caching** tells the API: "the prefix up to this breakpoint is reusable — store it on your side; next time I send the same prefix, charge me cache-read prices instead of input prices."

### Mental model

```
[ system prompt (10k tok, stable)  ][ tool defs (3k tok, stable) ][ user msg (200 tok, varies) ]
                                                                  ^ breakpoint here
```

The first call writes the cache (slightly *more* expensive than normal input). Subsequent calls within the cache TTL (5 minutes default, 1 hour available) read the cache at ~10% of input price.

### What to cache

- ✅ Long system prompts / personas
- ✅ Tool definitions (especially many tools)
- ✅ Few-shot examples
- ✅ Big retrieved documents that stay constant across follow-ups
- ❌ The user's current message (changes every call)
- ❌ Anything <1024 tokens (Haiku) / <2048 (Sonnet/Opus) — minimum cacheable size

### The mechanic

Put `cache_control={"type": "ephemeral"}` on the **last** content block of the segment you want cached. Everything *before* that block (system + earlier blocks) gets cached as a prefix.


In [ ]:
# 4) Caching demo — large stable system prompt, two short user turns
LARGE_SYSTEM_PROMPT = (
    "You are a senior compliance analyst. " * 800  # ~ a few thousand tokens
    + "\nAlways cite section numbers when answering."
)

def ask_cached(user_msg: str, tenant: str = "default"):
    return meter.messages_create(
        tag="cached_demo", tenant=tenant,
        model="claude-sonnet-4-5",
        max_tokens=200,
        system=[
            {
                "type": "text",
                "text": LARGE_SYSTEM_PROMPT,
                "cache_control": {"type": "ephemeral"},
            }
        ],
        messages=[{"role": "user", "content": user_msg}],
    )

print("--- Call 1 (writes cache) ---")
r1 = ask_cached("Summarize SOX 404 in two sentences.")
print("usage:", r1.usage)

print("\n--- Call 2 (reads cache) ---")
r2 = ask_cached("And what about HIPAA's minimum-necessary rule?")
print("usage:", r2.usage)

print("\n--- Cost breakdown so far ---")
print("by_tag:", meter.by_tag())


**What to look for:**

- Call 1's `usage` should show `cache_creation_input_tokens > 0` and `cache_read_input_tokens == 0`.
- Call 2 should show `cache_creation_input_tokens == 0` and `cache_read_input_tokens` matching the prefix size. The dollar cost of Call 2 will be a *fraction* of Call 1.

> ⚠️ **Common pitfall:** anything that changes inside the cached prefix invalidates it. Even a single different token at the start busts the cache. Keep the volatile bits (user query, current timestamp) at the *end*.

> 💡 **EXPERIMENT:** Add a second `cache_control` breakpoint between system and a long few-shot block. You can have up to 4 breakpoints; this lets you cache layered prefixes that change at different rates.


---
## 5) Message Batches API — 50% off, async

For workloads that don't need a real-time response — overnight evals, bulk reranking, dataset labeling, content moderation backlogs — Anthropic's **Message Batches API** runs your requests asynchronously and charges **half price** on everything (input, output, cache).

### Mental model

You hand the API a JSONL file of N independent requests. It returns a `batch_id`. Whenever it's done (usually minutes, max 24h), you download the results. SLA is best-effort, not interactive.

### When to use it

- ✅ Nightly eval runs against a labeled set
- ✅ Backfilling embeddings or classifications on historical data
- ✅ Generating synthetic training data
- ❌ Anything a user is waiting on
- ❌ Agent inner loops (latency-sensitive)


In [ ]:
# 5) Submitting a Message Batch (illustrative — comment out the wait if running live)
from anthropic.types.messages.batch_create_params import Request

BATCH_MODEL = "claude-haiku-4-5-20251001"

batch_requests = [
    Request(
        custom_id=f"task-{i}",
        params={
            "model": BATCH_MODEL,
            "max_tokens": 80,
            "messages": [{"role": "user", "content": q}],
        },
    )
    for i, q in enumerate([
        "Classify sentiment: 'This product changed my life.'",
        "Classify sentiment: 'It's fine I guess.'",
        "Classify sentiment: 'Hard pass.'",
        "Classify sentiment: 'Best purchase of the year.'",
        "Classify sentiment: 'Worked for two days then died.'",
    ])
]

batch = client.messages.batches.create(requests=batch_requests)
print("Submitted batch:", batch.id, "status:", batch.processing_status)

# Polling (small batches usually finish in <1 min). Skip this loop if you want to come back later.
import time
for _ in range(30):
    b = client.messages.batches.retrieve(batch.id)
    if b.processing_status == "ended":
        break
    print("…", b.processing_status)
    time.sleep(5)

# Stream results
print("\nResults:")
for line in client.messages.batches.results(batch.id):
    if line.result.type == "succeeded":
        msg = line.result.message
        print(f"{line.custom_id}: {msg.content[0].text}  usage={msg.usage}")
    else:
        print(f"{line.custom_id}: FAILED — {line.result}")


**Accounting note:** the per-request `usage` you get back is the same shape as a normal call. To bill it correctly through your `CostMeter`, pass `batched=True`. Tip: in production, write a thin `submit_batch_with_meter()` helper that ingests the results stream and creates one `CallRecord` per row.

> 💡 **EXPERIMENT:** Run the same 5 prompts as five individual `messages.create()` calls and compare total spend vs. the batched run. ~50% delta.

> ⚠️ **Pitfall:** batches are async — your code must be fine with delayed results. Don't batch anything an end-user is staring at.


---
## 6) Token compression — paying for signal, not noise

You can spend less by *sending* less. Three practical techniques, ranked by safety:

### a) Drop low-information content (safe)
- Strip HTML/markdown chrome from scraped pages before feeding them in.
- Remove repeated boilerplate (headers, footers, nav).
- De-duplicate near-identical retrieved chunks.

### b) Summarize stale context (mostly safe)
- In a long agent loop, every ~N turns replace the oldest messages with an LLM-generated summary. (You did this in Lesson 5 — Agent Memory.)
- For RAG, summarize each retrieved doc *before* stuffing all of them into the prompt.

### c) Lossy structural compression (use with care)
- Reformat verbose JSON as compact JSON (no whitespace).
- Replace long enums with short codes (`status=PROCESSING_INVENTORY_RECONCILIATION` → `status=PIR`).
- Use **LLMLingua / Selective Context** style compressors that drop low-perplexity tokens — measure quality before/after.

Let's implement (a) and (b) and measure.


In [ ]:
# 6) Cheap, deterministic compressors
import re, tiktoken

# Token counting (rough — Anthropic uses its own tokenizer, but tiktoken is a fine estimator)
enc = tiktoken.get_encoding("cl100k_base")
def n_tokens(s: str) -> int:
    return len(enc.encode(s))

# (a) Drop low-info content
def strip_chrome(html_or_md: str) -> str:
    s = re.sub(r"<script.*?</script>", " ", html_or_md, flags=re.S | re.I)
    s = re.sub(r"<style.*?</style>", " ", s, flags=re.S | re.I)
    s = re.sub(r"<[^>]+>", " ", s)                  # strip tags
    s = re.sub(r"!\[[^\]]*\]\([^)]+\)", " ", s) # markdown images
    s = re.sub(r"\[[^\]]+\]\([^)]+\)", lambda m: m.group(0).split("]")[0][1:], s)  # keep link text only
    s = re.sub(r"\s+", " ", s).strip()
    return s

# (b) Summarize stale chunks with Haiku (cheap)
def summarize(text: str, max_words: int = 80) -> str:
    resp = meter.messages_create(
        tag="compress_summarize", tenant="default",
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        system=f"Summarize the user's text faithfully in <= {max_words} words. "
               f"Preserve named entities, numbers, dates, and any explicit claims.",
        messages=[{"role": "user", "content": text}],
    )
    return resp.content[0].text.strip()

# Demo
RAW = """<html><head><script>track();</script></head><body>
<nav>Home · Pricing · Docs</nav>
<article>Acme Corp announced Q1 revenue of $2.1B (up 17% YoY) on April 21, 2026.
Operating margin held at 22%. CEO Lina Chen attributed growth to enterprise contracts
in the Asia-Pacific region, particularly Japan and South Korea, where bookings
doubled year-over-year. The company reiterated full-year guidance of $9–9.4B revenue.
A new manufacturing facility in Penang will come online in Q3.</article>
<footer>© 2026 Acme · Privacy · Terms</footer></body></html>"""

stripped = strip_chrome(RAW)
summary = summarize(stripped)
print(f"raw:       {n_tokens(RAW):4d} tokens")
print(f"stripped:  {n_tokens(stripped):4d} tokens")
print(f"summary:   {n_tokens(summary):4d} tokens  ← what you actually feed to the next call")
print("\nSummary text:\n", summary)


> 💡 **EXPERIMENT:** Compute the **break-even** for summarization. Summarizing costs Haiku tokens. If you'll reuse the summary across ≥ K downstream calls on Sonnet, summarization pays off. Derive K for your traffic.

> ⚠️ **Pitfall:** compression is **lossy**. Always have an eval that catches "the model started giving wrong answers because we squeezed too hard." Cost wins mean nothing if quality silently drops.


---
## 7) Per-tenant cost observability

In a multi-tenant SaaS you need to know: **whose spend is this?** Three reasons:

1. **Billing** — pass usage through (with margin) to customers.
2. **Limits** — soft-cap a tenant once they cross $X/day so a bug doesn't bankrupt you.
3. **Product analytics** — which features are net-positive after their LLM costs?

Our `CostMeter` already records `tenant` and `tag`. Let's add a rate limiter and a small dashboard.


In [ ]:
# 7) Per-tenant analytics + a soft cap
class TenantCapExceeded(Exception):
    pass

class TenantAwareMeter(CostMeter):
    def __init__(self, client, daily_cap_usd: dict[str, float] | None = None):
        super().__init__(client)
        self.daily_cap = daily_cap_usd or {}

    def messages_create(self, **kwargs):
        tenant = kwargs.get("tenant", "default")
        cap = self.daily_cap.get(tenant)
        if cap is not None and self.by_tenant().get(tenant, 0.0) >= cap:
            raise TenantCapExceeded(
                f"Tenant {tenant!r} hit daily cap ${cap:.4f} "
                f"(current spend ${self.by_tenant().get(tenant, 0.0):.4f})"
            )
        return super().messages_create(**kwargs)

tmeter = TenantAwareMeter(client, daily_cap_usd={"free_user_42": 0.005, "enterprise_acme": 5.00})

# Simulate some traffic
def run_one(tenant: str, q: str):
    try:
        resp = tmeter.messages_create(
            tag="qa", tenant=tenant,
            model="claude-haiku-4-5-20251001",
            max_tokens=120,
            messages=[{"role": "user", "content": q}],
        )
        return resp.content[0].text[:60] + "…"
    except TenantCapExceeded as e:
        return f"BLOCKED: {e}"

for t in ["enterprise_acme", "enterprise_acme", "free_user_42",
          "free_user_42", "free_user_42", "free_user_42"]:
    print(f"[{t:18s}] {run_one(t, 'Give me three startup ideas in supply-chain SaaS.')}")

print("\nSpend by tenant:", tmeter.by_tenant())


In [ ]:
# 7b) Quick chart — spend by tag
import matplotlib.pyplot as plt
df = tmeter.as_df()
if not df.empty:
    by_tag = df.groupby("tag")["cost_usd"].sum().sort_values()
    by_tag.plot(kind="barh", title="Spend by tag ($)")
    plt.xlabel("USD")
    plt.tight_layout()
    plt.show()
else:
    print("(No records yet)")


> 💡 **EXPERIMENT:** lower `free_user_42`'s cap to `$0.001` and watch a single call trip the breaker. In production this same pattern protects you from runaway-loop bugs.

> 🧠 **Where this lives in real systems:** the `CostMeter` is your in-process collector. In production you'd ship records to a time-series store (e.g. ClickHouse, BigQuery, Postgres) and overlay them with traces (Lesson 8 — observability). The wrapper interface stays the same.


---
## 8) Mini-capstone — `CostAwareAutoResearcher`

A tiny end-to-end agent that wires all five levers together. Given a research question, it:

1. **Routes**: classifier decides "shallow lookup" vs. "deep research."
2. **Caches**: a long, stable researcher persona is sent with `cache_control`.
3. **Compresses**: retrieved docs are stripped + summarized via Haiku before going into the Sonnet prompt.
4. **Batches** *(optional toggle)*: when given a list of N questions, can submit them as a batch instead of looping.
5. **Bills**: every call attributed to a `tenant`, cost recorded.

Wire it up, run a few questions, look at the cost breakdown.


In [ ]:
# 8) CostAwareAutoResearcher
RESEARCHER_PERSONA = (
    "You are AutoResearcher, a careful research assistant.\n"
    "- Always cite which sources you used.\n"
    "- Distinguish 'verified from sources' vs 'inferred'.\n"
    "- If sources conflict, say so.\n"
    "- Be concise: ≤ 150 words unless asked.\n"
    + ("FILLER NOTE: " + "x " * 1500)  # pad to make the cache meaningful
)

FAKE_SOURCES = [
    "<html><body><nav>nav</nav><article>Acme Q1 2026 revenue $2.1B, +17% YoY, op margin 22%. New Penang plant Q3 2026.</article><footer>©</footer></body></html>",
    "<html><body><article>Analyst note (Apr 22, 2026): Acme's APAC bookings doubled YoY led by Japan/SK. FY guide $9–9.4B reiterated.</article></body></html>",
    "<html><body><article>Competitor BetaCo posted Q1 revenue $1.4B (+8%), margin 14%, citing supply-chain headwinds.</article></body></html>",
]

def cost_aware_research(question: str, tenant: str = "default") -> str:
    # 1) Route
    difficulty = classify(question, tenant=tenant)  # uses Haiku
    answer_model = ROUTING_MODELS[difficulty]

    # 3) Compress retrieved docs once (cheap with Haiku)
    cleaned = [strip_chrome(d) for d in FAKE_SOURCES]
    summaries = [summarize(c, max_words=40) for c in cleaned]
    context = "\n\n".join(f"[S{i+1}] {s}" for i, s in enumerate(summaries))

    # 2) Final call with cache_control on the persona
    resp = meter.messages_create(
        tag=f"final_{difficulty}", tenant=tenant,
        model=answer_model,
        max_tokens=350,
        system=[{
            "type": "text",
            "text": RESEARCHER_PERSONA,
            "cache_control": {"type": "ephemeral"},
        }],
        messages=[{
            "role": "user",
            "content": f"Sources:\n{context}\n\nQuestion: {question}",
        }],
    )
    return resp.content[0].text.strip()

# Run a few
for q, tenant in [
    ("What was Acme's Q1 2026 revenue?", "enterprise_acme"),
    ("Compare Acme and BetaCo's Q1 2026 performance and explain the margin gap.", "enterprise_acme"),
    ("When does Penang come online?", "free_user_42"),
]:
    print(f"\nQ ({tenant}): {q}")
    print("A:", cost_aware_research(q, tenant=tenant))

print("\n--- Cost report ---")
print("Total      :", f"${meter.total():.6f}")
print("By tenant  :", meter.by_tenant())
print("By tag     :", meter.by_tag())


**Look at the `by_tag` breakdown.** You should see most of the spend in `final_*` calls (the answer step on Sonnet) and a small slice in `route_classify` + `compress_summarize` (the Haiku scaffolding). That's the shape of a healthy routed system: the expensive model only handles the bit that needs it.

### 🧪 EXPERIMENT (graded by you)

1. Disable caching (remove the `cache_control` block) and re-run. Compare total spend.
2. Force every question through Sonnet (skip the router). Compare total spend AND quality on the easy question.
3. Add a second user turn ("What about their FY guide?") in the same loop with the same persona. Confirm cache hit (`cache_read_input_tokens > 0`).
4. Wire the batch API into a `cost_aware_research_many(questions)` function that submits all final answers as a single batch. Half-price the most expensive part.


---
## 9) Recap

You can now:

- Read an Anthropic `usage` object and translate it into dollars with the right discounts applied.
- Instrument every call with a `CostMeter` that tracks model, tag, tenant, latency, and cost.
- Route easy work to Haiku and hard work to Sonnet/Opus — and know **when not to** (quality risk without evals).
- Use `cache_control` breakpoints to cut up to 90% off repeated long prefixes.
- Submit asynchronous **Message Batches** for non-realtime work at half price.
- Compress noisy context losslessly-ish before feeding the expensive model.
- Attribute spend per tenant and enforce soft daily caps.

### Cost-engineering mental model (memorize this)

```
Total $ = (calls × per_call_input_tokens × input_price)
       + (calls × per_call_output_tokens × output_price)
       − (cache_hits × cached_tokens × 90%)
       − (batched_calls × everything × 50%)
       − (routed_easy × (big_model_price − small_model_price))
       − (compressed_calls × dropped_tokens × input_price)
```

Every lever in this lesson is a term in that equation.

### Production checklist

- [ ] Every LLM call goes through a tracker that records `tenant` + `tag` + `cost`.
- [ ] You have an eval suite (Lesson 17) that catches quality regressions from routing/compression.
- [ ] At least one stable, long prefix is cached.
- [ ] Non-realtime workloads (evals, backfills) go through the Batch API.
- [ ] Soft daily caps per tenant prevent runaway-loop disasters.
- [ ] A weekly $ dashboard exists; someone is on the hook for watching it.

---

## 🎯 Next lesson (23 — Phase 3 Capstone)

You ship the real thing. **AutoResearcher v1.0**, end-to-end:
- Deployed (Lesson 16) on FastAPI + Docker behind an API key.
- Evaluated (Lesson 17) with a CI gate.
- Secured (Lesson 18) with input/output guardrails.
- Streamed (Lesson 19) over SSE to a tiny UI.
- Backed by a real vector DB (Lesson 20).
- Implemented with a framework of your choice (Lesson 21).
- **Cost-engineered (this lesson) with routing + caching + per-tenant attribution.**

We'll package it as a public GitHub repo + PyPI release so it's the open-source artifact that proves your AI engineering chops.

See you tomorrow. 🚀
